In [17]:
import vcfpy
import pandas as pd
import requests
import json


In [5]:
vcf_reader = vcfpy.Reader(open('challenge_data.vcf', 'r'))

In [30]:
record = next(vcf_reader)

chrom = record.CHROM
pos = record.POS
ref = record.REF
alt = record.ALT[0] # ",".join(str(_alt) for _alt in record.ALT)

# 1. "DP": Depth of sequence coverage at site of variation
depth = record.INFO.get('DP', None)
# 2. "AD": Number of reads supporting the variant
allelic_depth = record.INFO.get('AD', None)

print(f"""CHROMOSOME: {chrom}
POSITION: {pos}
REFERENCE: {ref}
ALTERNATE: {alt}
DEPTH: {depth}
ALLELIC DEPTH: {allelic_depth}
""")

# 4. Query Ensemble VEP API
vep_params = {
    # 'region': f'{chrom}:{pos}:{pos}/{alt.value}',
    # "allele": f"{ref}/{alt.value}",
    # 'variants': f'{chrom} {pos} {ref} {alt}',
    'content-type': 'application/json'
}
# vep_payload = {
#     'variants': [f'{chrom} {pos} {ref} {alt}']
# }
VEP_API_URL = f"https://grch37.rest.ensembl.org/vep/human/region/{chrom}:{pos}/{alt.value}"

try:
    response = requests.get(VEP_API_URL, params=vep_params)
    # response = requests.post(VEP_API_URL, headers=vep_params, data=json.dumps(vep_payload))
    print(response.json())
except Exception as e:
    print(e)

CHROMOSOME: 1
POSITION: 1654129
REFERENCE: TAAAAAAAT
ALTERNATE: Substitution(type_='INDEL', value='TAAAAAAT')
DEPTH: 2198
ALLELIC DEPTH: None

[{'id': '1_1654130_-/AAAAAAT', 'seq_region_name': '1', 'most_severe_consequence': '5_prime_UTR_variant', 'assembly_name': 'GRCh37', 'input': '1 1654129 1654129 T/TAAAAAAT 1', 'start': 1654130, 'allele_string': '-/AAAAAAT', 'end': 1654129, 'strand': 1, 'transcript_consequences': [{'variant_allele': 'AAAAAAT', 'strand': -1, 'gene_symbol_source': 'HGNC', 'distance': 2147, 'gene_symbol': 'SLC35E2', 'hgnc_id': 20863, 'consequence_terms': ['downstream_gene_variant'], 'gene_id': 'ENSG00000215790', 'biotype': 'protein_coding', 'transcript_id': 'ENST00000355439', 'impact': 'MODIFIER'}, {'variant_allele': 'AAAAAAT', 'gene_symbol_source': 'HGNC', 'strand': -1, 'hgnc_id': 1730, 'gene_symbol': 'CDK11A', 'cdna_start': 141, 'biotype': 'protein_coding', 'consequence_terms': ['5_prime_UTR_variant'], 'gene_id': 'ENSG00000008128', 'cdna_end': 142, 'transcript_id':